# Imports

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data

In [2]:
appart_df = pd.read_csv('datasets_prepd/dvf_appartement.csv')
print(appart_df.head())

    id_mutation date_mutation  numero_disposition nature_mutation  \
0  2021-1289506    2021-01-08                   1           Vente   
1  2021-1289527    2021-01-11                   1           Vente   
2  2021-1289529    2021-01-04                   1           Vente   
3  2021-1289537    2021-01-06                   1           Vente   
4  2021-1289546    2021-01-04                   1           Vente   

   valeur_fonciere  adresse_numero adresse_suffixe  \
0         165000.0             1.0             NaN   
1         167000.0            11.0             NaN   
2         166000.0            51.0             NaN   
3         180000.0            66.0             NaN   
4         120000.0            50.0             NaN   

             adresse_nom_voie adresse_code_voie  code_postal  ...  \
0        RUE DU MOULIN A VENT              1570      77127.0  ...   
1  PL ANTOINE DE BOUGAINVILLE              0124      77380.0  ...   
2              RUE SOMMEVILLE              0960      

C:\Users\Adam\AppData\Local\Temp\ipykernel_26432\2213143999.py:1: DtypeWarning: Columns (0: lot2_numero) have mixed types. Specify dtype option on import or set low_memory=False.
  appart_df = pd.read_csv('datasets_prepd/dvf_appartement.csv')


# Features

In [3]:
features = [
    "longitude",
    "latitude",
    "code_postal",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "prix_m2_ref",
    "total_carrez_surface",
    "number_of_lots",
    "season"
]

target = "valeur_fonciere_actualisee"

# Encode season to numeric values
season_mapping = {"winter": 0, "spring": 1, "summer": 2, "autumn": 3}
appart_df["season"] = appart_df["season"].map(season_mapping)

X = appart_df[features]
y = appart_df[target]

print("Feature count:", X.shape[1])
print(X.head())
print(y.head())

Feature count: 9
   longitude   latitude  code_postal  surface_reelle_bati  \
0   2.553326  48.628821      77127.0                 66.0   
1   2.569526  48.656643      77380.0                 67.0   
2        NaN        NaN      77380.0                 81.0   
3   2.562435  48.665468      77380.0                 60.0   
4   2.804714  48.376407      77250.0                 41.0   

   nombre_pieces_principales   prix_m2_ref  total_carrez_surface  \
0                        3.0   4588.322776                 66.22   
1                        3.0  25488.264429                 67.40   
2                        4.0  25488.264429                 81.52   
3                        3.0  25488.264429                 60.06   
4                        2.0  12691.832995                 41.99   

   number_of_lots  season  
0             1.0       0  
1             1.0       0  
2             1.0       0  
3             1.0       0  
4             1.0       0  
0    165000.0
1    167000.0
2    166000

# Make sets : trail and test and validate

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)

Train size: (18620, 9)
Validation size: (3990, 9)
Test size: (3991, 9)


# Ssave

In [5]:
os.makedirs("data_apt", exist_ok=True)

train_df = X_train.copy()
val_df = X_val.copy()
test_df = X_test.copy()

train_df[target] = y_train
val_df[target] = y_val
test_df[target] = y_test

train_df.to_csv("data_apt/appart_train.csv", index=False)
val_df.to_csv("data_apt/appart_val.csv", index=False)
test_df.to_csv("data_apt/appart_test.csv", index=False)

print("Datasets saved.")

Datasets saved.


# Train model

In [6]:
model_appart = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

model_appart.fit(X_train, y_train)

print("Model trained.")

Model trained.


# Evaluation

In [7]:
y_val_pred = model_appart.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
mae = mean_absolute_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

print("Validation RMSE:", rmse)
print("Validation MAE:", mae)
print("Validation R2:", r2)

Validation RMSE: 37692.80983299524
Validation MAE: 25134.634153985393
Validation R2: 0.7799533383160284


SAVE MODEL

In [8]:
os.makedirs("models", exist_ok=True)

joblib.dump(model_appart, "models/apt_random_forest_model.pkl")

print("Model saved.")

Model saved.


In [9]:
print("Number of features expected:", model_appart.n_features_in_)
print("Feature names:", features)

Number of features expected: 9
Feature names: ['longitude', 'latitude', 'code_postal', 'surface_reelle_bati', 'nombre_pieces_principales', 'prix_m2_ref', 'total_carrez_surface', 'number_of_lots', 'season']
